# pydeb-bench — interactive demo

Companion software for Hackenberger & Djerdj (2026), *Ecological Modelling*.

This notebook is the Binder landing page: it fits a simplified Dynamic Energy
Budget (DEB) growth model to synthetic *Daphnia magna* data using three
paradigms — classical least-squares, Bayesian MCMC with AmP-informed priors,
and Random Forest — and produces four of the six publication-quality
figures from `pydeb.plots`.

To run everything, use the menu: **Run → Run All Cells**.
Expect ~30 s total on the Binder server.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from debcompare import simulate_daphnia_dataset
from pydeb.bayes import classical_fit, fit_growth, summarise
from pydeb.ml.baselines import fit_random_forest
from pydeb.plots import (
    plot_arrhenius,
    plot_prior_posterior,
    plot_posterior_corner,
    plot_posterior_predictive_fan,
)

## 1. Simulate growth data

15 time points × 5 replicates at 20 °C (training) and 25 °C (extrapolation
test), using AmP-derived ground-truth parameters for *Daphnia magna*.

In [ ]:
ds = simulate_daphnia_dataset(random_seed=42)
print(f'Training set: {len(ds.train)} observations at {ds.train["temp"].iloc[0]:.0f} °C')
print(f'Test set:     {len(ds.test)} observations at {ds.test["temp"].iloc[0]:.0f} °C')
print(f'True params:  {ds.true_params}')

## 2. Classical DEB: Nelder–Mead least-squares

In [ ]:
cf = classical_fit(ds.train, T_A=ds.true_params.T_A, T_ref_C=ds.true_params.T_ref_C)
print(cf.summary())

## 3. Bayesian DEB: NUTS with AmP-informed priors

Reduced draws (800 × 2 chains) for Binder's modest compute. Production
benchmark uses 2000 × 3. Both converge to R-hat = 1.00.

In [ ]:
idata = fit_growth(
    ds.train,
    draws=800, tune=800, chains=2,
    random_seed=42, progressbar=False,
)
summarise(idata)

## 4. Random Forest baseline

In [ ]:
from debcompare.metrics import rmse
rf = fit_random_forest(ds.train)
print(f'RF interpolation RMSE: {rmse(ds.train["L_obs"], rf.predict(ds.train)):.3f} mm')
print(f'RF extrapolation RMSE: {rmse(ds.test["L_obs"], rf.predict(ds.test)):.3f} mm')
print('Paper claim replicated: RF extrapolation substantially worse than interpolation.')

## 5. Educational figures

### 5a. Arrhenius temperature correction

Mechanistic handling of temperature — the reason DEB extrapolates
where Random Forest cannot.

In [ ]:
fig = plot_arrhenius(ds.true_params)
plt.show()

### 5b. Prior vs posterior

How much information the data contributed to each parameter.

In [ ]:
fig = plot_prior_posterior(idata)
plt.show()

### 5c. Joint posterior (corner plot)

The $L_\infty$–$r_B$ anti-correlation is the visible signature of
equifinality discussed in §3.4 of the paper.

In [ ]:
fig = plot_posterior_corner(idata)
plt.show()

### 5d. Posterior predictive fan across temperatures

Training data exists only at 20 °C; the other four curves are pure
mechanistic extrapolation via the Arrhenius correction.

In [ ]:
fig = plot_posterior_predictive_fan(
    idata,
    params=ds.true_params,
    train_data=ds.train,
    n_samples=200,
)
plt.show()

## Next steps

- Run the full reproducible benchmark (CLI): in a terminal,
  `python -m debcompare --gallery -o benchmark_output`
- See `README.md` for the full API.
- See `software_description.pdf` for the companion document describing
  purpose, architecture, and how each module maps to specific claims in
  the accompanying review.

**Citation.** If you use this software, please cite:

> Hackenberger, B.K. & Djerdj, T. (2026). *Mechanistic, Bayesian, and
> Machine Learning Approaches to Metabolic Modelling: A Critical Review
> of Dynamic Energy Budget Theory and Its Alternatives.*
> Ecological Modelling (in press).